# Coulomb Archive 검증 실험

`docs/coulomb-archive.md` 의 두 규칙이 실제로 다음을 만족하는지 검증한다.

1. **Streaming 아카이빙**: 자식이 1개씩 들어올 때 배치 재선택 없이 유지되는가
2. **Near-duplicate 축출**: 조건절 비대화(H4/H5), AND 교환(H6/H7) 페어에서 하나만 살아남는가
3. **논리 대비 보존**: 인과 역전(H2/H3), 관계 유형 반전(H10/H11) 페어는 둘 다 살아남는가
4. **부모 샘플링 편향**: 붐빈 지역의 부모 선택 확률이 실제로 낮아지는가
5. **$\gamma$ 하나로 exploration-exploitation 조절 가능한가**

베이스라인: **score-only top-K** (Coulomb 규칙을 끄고 quality 만 보고 유지).

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / 'hypoevolve').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt

from hypoevolve.elg.ir import (
    AtomicNode, LogicalNode, RelationNode,
    LogicalOp, RelationType,
)

np.random.seed(0)
print(f'Repo root: {repo_root}')

## 1. ELG 가설 12개 정의

DPP-Elites 실험과 동일한 population 을 재사용. Near-duplicate 페어 두 개(H4/H5, H6/H7) 와 논리 대비 페어 세 개(H2/H3, H8/H9, H10/H11) 를 포함.

In [ ]:
def A(name): return AtomicNode(name=name)
def AND(*c): return LogicalNode(name=LogicalOp.AND, inputs=list(c))
def NOT(c):  return LogicalNode(name=LogicalOp.NOT, inputs=[c])
def IMPLIES(a, b):    return RelationNode(name=RelationType.IMPLIES, inputs=[a, b])
def CONTRADICT(a, b): return RelationNode(name=RelationType.CONTRADICT, inputs=[a, b])
def CORRELATE(a, b):  return RelationNode(name=RelationType.CORRELATE, inputs=[a, b])

hypotheses = {
    'H0_stock_baseline':   IMPLIES(A('stock A price above 20-day moving average'),
                                   A('stock A returns positive next day')),
    'H1_weather_baseline': IMPLIES(A('temperature exceeds 30 degrees'),
                                   A('ice cream sales rise')),
    'H2_causal_ref':       IMPLIES(A('company earnings beat estimates'),
                                   A('stock price rises')),
    'H3_causal_rev':       IMPLIES(A('stock price rises'),
                                   A('company earnings beat estimates')),
    'H4_clean':            IMPLIES(A('VIX above 30'),
                                   A('SPY drops next day')),
    'H5_cluttered':        IMPLIES(AND(A('VIX above 30'),
                                       A('if it is a weekday'),
                                       A('if no earnings announcement'),
                                       A('if trading volume is normal')),
                                   A('SPY drops next day')),
    'H6_and_ref':          IMPLIES(AND(A('RSI below 30'), A('MACD negative')),
                                   A('buy signal triggered')),
    'H7_and_swap':         IMPLIES(AND(A('MACD negative'), A('RSI below 30')),
                                   A('buy signal triggered')),
    'H8_pos':              IMPLIES(A('momentum strong'),
                                   A('price up next week')),
    'H9_neg':              IMPLIES(A('momentum strong'),
                                   NOT(A('price up next week'))),
    'H10_correlate':       CORRELATE(A('temperature high'),
                                     A('cold drink sales rise')),
    'H11_contradict':      CONTRADICT(A('temperature high'),
                                      A('cold drink sales rise')),
}

quality = {
    'H0_stock_baseline':   0.55,
    'H1_weather_baseline': 0.55,
    'H2_causal_ref':       0.78,
    'H3_causal_rev':       0.72,
    'H4_clean':            0.90,
    'H5_cluttered':        0.85,  # deceptively high (near-duplicate of H4)
    'H6_and_ref':          0.80,
    'H7_and_swap':         0.75,  # near-duplicate of H6
    'H8_pos':              0.70,
    'H9_neg':              0.50,
    'H10_correlate':       0.55,
    'H11_contradict':      0.50,
}
names = list(hypotheses.keys())
hs = [hypotheses[n] for n in names]
q  = np.array([quality[n] for n in names])
N  = len(hs)
print(f'N = {N} hypotheses defined')

## 2. 거리: $d(h_i, h_j) = 1 - K_{\\text{tree}}(h_i, h_j)$

Tree kernel은 두 가지 upgrade를 반영한다.

**Soft Jaccard on children (AND / OR)**: 자식 매칭을 단순 sum이 아니라 자카드로 정의. 남는 자식이 명시적으로 합집합에 포함되어 유사도를 낮춤.

$$K_{\\text{AND/OR}}(c_1, c_2) = \\frac{\\text{intersection}(c_1, c_2)}{|c_1| + |c_2| - \\text{intersection}(c_1, c_2)}$$

**Wrapper Descent (kind mismatch)**: `atomic` vs `AND/OR`처럼 kind가 다를 때, 통째로 0으로 처리하지 않고 wrapper를 벗겨 내부 자식과 비교하되 decay $\\lambda_{\\text{wrap}} = 0.5$ 곱. NOT wrapper는 극성 페널티 $\\lambda_{\\text{neg}} = 0.3$.

**Relation nodes**: `IMPLIES/CONTRADICT/CORRELATE` 자식(condition, target)의 유사도 평균. 위치 순서에 민감.

트리 커널 값이 이미 [0, 1] 범위이므로 별도 정규화 나눗셈 불필요. 거리 $d = 1 - K$.

In [ ]:
def ngrams(s, n=3):
    s = s.lower()
    if len(s) < n: return {s}
    return set(s[i:i+n] for i in range(len(s) - n + 1))

def atomic_sim(s1, s2):
    a, b = ngrams(s1), ngrams(s2)
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return len(a & b) / len(a | b)

# Decay factors for kind-mismatch wrapper descent
LAMBDA_WRAP = 0.5   # atomic vs AND/OR: descend into wrapper's children
LAMBDA_NEG  = 0.3   # NOT wrapper: polarity flip penalty

def _soft_jaccard(cs1, cs2, kfn):
    """Soft Jaccard on children: intersection / union.
    intersection = greedy best-match sum,  union = |cs1| + |cs2| − intersection.
    Extra unmatched children explicitly INCREASE the union → lower similarity."""
    if not cs1 and not cs2: return 1.0
    if not cs1 or not cs2:  return 0.0
    import numpy as np
    sims = np.array([[kfn(c1, c2) for c2 in cs2] for c1 in cs1])
    triples = sorted([(sims[i, j], i, j) for i in range(len(cs1)) for j in range(len(cs2))],
                     key=lambda x: -x[0])
    inter = 0.0
    used_r, used_c = set(), set()
    for v, i, j in triples:
        if i in used_r or j in used_c: continue
        inter += v
        used_r.add(i); used_c.add(j)
    union = len(cs1) + len(cs2) - inter
    return inter / union if union > 1e-9 else 0.0

def _wrapper_descent(t1, t2):
    """Kind mismatch: instead of returning 0 flat, descend through wrappers with decay."""
    # atomic on one side, AND/OR wrapper on the other → best child match with LAMBDA_WRAP
    if t1.kind == 'atomic' and t2.kind == 'logical' and t2.name in (LogicalOp.AND, LogicalOp.OR):
        best = max((tree_kernel(t1, c) for c in t2.inputs), default=0.0)
        return LAMBDA_WRAP * best
    if t2.kind == 'atomic' and t1.kind == 'logical' and t1.name in (LogicalOp.AND, LogicalOp.OR):
        best = max((tree_kernel(c, t2) for c in t1.inputs), default=0.0)
        return LAMBDA_WRAP * best
    # NOT wrapper vs anything else: recurse into NOT's child with polarity penalty
    if t1.kind == 'logical' and t1.name == LogicalOp.NOT:
        return LAMBDA_NEG * tree_kernel(t1.inputs[0], t2)
    if t2.kind == 'logical' and t2.name == LogicalOp.NOT:
        return LAMBDA_NEG * tree_kernel(t1, t2.inputs[0])
    return 0.0

def tree_kernel(t1, t2):
    """Return similarity in [0, 1] directly. No external normalization needed."""
    if t1.kind != t2.kind:
        return _wrapper_descent(t1, t2)
    if t1.kind == 'atomic':
        return atomic_sim(t1.name, t2.name)
    if t1.kind == 'logical':
        if t1.name != t2.name: return 0.0
        if t1.name == LogicalOp.NOT:
            return tree_kernel(t1.inputs[0], t2.inputs[0])
        # AND / OR: soft Jaccard on children — extra children raise the union
        return _soft_jaccard(t1.inputs, t2.inputs, tree_kernel)
    if t1.kind == 'relation':
        if t1.name != t2.name: return 0.0
        # Fixed-position children: average of condition and target similarities
        cond   = tree_kernel(t1.inputs[0], t2.inputs[0])
        target = tree_kernel(t1.inputs[1], t2.inputs[1])
        return 0.5 * (cond + target)
    return 0.0

# Convenience aliases so downstream cells (K matrix, distance) stay unchanged
def K_norm(t1, t2):   return tree_kernel(t1, t2)
def distance(t1, t2): return 1.0 - tree_kernel(t1, t2)

# Pre-compute pairwise similarity / distance matrices for the 12 hypotheses
import numpy as np
K = np.zeros((N, N))
D = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        K[i, j] = tree_kernel(hs[i], hs[j])
        D[i, j] = 1.0 - K[i, j]

print('Cross-pair distances (selected) — new kernel (Soft Jaccard + Wrapper Descent):')
for a, b in [(4, 5), (6, 7), (2, 3), (8, 9), (10, 11), (0, 1)]:
    label = f'{names[a]:<20} <-> {names[b]:<20}'
    print(f'  d={D[a, b]:.3f}   K={K[a, b]:.3f}   {label}')


## 3. Coulomb Potential $U(h; A)$

$$U(h; A) = \sum_{h' \in A,\, h' \neq h} \frac{\text{score}(h')}{d(h, h')^2 + \epsilon}$$

각 아카이브 멤버가 만드는 반발을 선형 중첩. 이 값이 그 위치의 '혼잡도'.

In [ ]:
EPS = 1e-2

def potential(h_idx, archive_idx, scores=q, eps=EPS):
    total = 0.0
    for j in archive_idx:
        if j == h_idx:
            continue
        d = D[h_idx, j]
        total += scores[j] / (d * d + eps)
    return total

# Sanity: potential of each hypothesis assuming the OTHER 11 are archived
all_others = lambda i: [j for j in range(N) if j != i]
U_all = np.array([potential(i, all_others(i)) for i in range(N)])

header = 'U(h; A minus h)'
print(f'{"hypothesis":<24}{"quality":>10}{header:>18}')
for i in range(N):
    print(f'{names[i]:<24}{q[i]:>10.2f}{U_all[i]:>15.2f}')
print()
print(f'Highest U (most crowded): {names[np.argmax(U_all)]}')
print(f'Lowest  U (most isolated): {names[np.argmin(U_all)]}')

**해석**: 붐빈 지역(near-duplicate 파트너가 있는 H4, H5, H6, H7)의 U 는 격리된 지역(H0, H1, H10, H11)보다 확연히 높아야 정상.

## 4. 규칙 1 — 부모 샘플링 확률

$$P(\text{parent} = h) \propto \text{score}(h) \cdot \exp(-\gamma \cdot U(h; A))$$

Coulomb 반발이 얼마나 부모 선택을 재분배하는지 관찰.

In [ ]:
def sampling_probs(archive_idx, gamma, scores=q):
    logits = np.full(N, -np.inf)
    for i in archive_idx:
        u = potential(i, archive_idx, scores)
        logits[i] = np.log(scores[i] + 1e-9) - gamma * u
    m = np.max(logits)
    logits = logits - m
    p = np.exp(logits)
    p[np.isinf(logits) | np.isnan(logits)] = 0.0
    return p / (p.sum() + 1e-12)

# When all 12 are in the archive, compare score-only vs Coulomb for various gamma
full_archive = list(range(N))
p_score_only = q / q.sum()
gammas_demo = [0.0, 0.05, 0.2, 1.0]

fig, ax = plt.subplots(figsize=(13, 5))
width = 0.18
x = np.arange(N)
for k, gamma in enumerate(gammas_demo):
    p = sampling_probs(full_archive, gamma)
    ax.bar(x + (k - 1.5) * width, p, width, label=f'γ = {gamma}')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=90, fontsize=8)
ax.set_ylabel('P(parent = h)')
ax.set_title('Sampling distribution over the full population as γ increases')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

# Print top-3 by sampling probability at gamma=0 vs gamma=1
print('Top-3 by sampling probability:')
for gamma in gammas_demo:
    p = sampling_probs(full_archive, gamma)
    top3 = np.argsort(-p)[:3]
    print(f'  γ = {gamma:<5}: ' + ', '.join(f'{names[i]}({p[i]:.3f})' for i in top3))

**기대**: $\gamma$ 가 커지면 H4, H5, H6, H7 처럼 서로 인접한 near-duplicate 페어들의 선택 확률이 낮아지고, H0, H1, H10, H11 같은 격리된 가설들의 확률이 상대적으로 올라감.

## 5. 규칙 2 — Streaming 아카이빙

$$\Delta(h^*) = \text{score}(h^*) - \gamma \cdot U(h^*; A)$$

- $\Delta > \tau$ 이면 편입
- 아카이브 크기가 상한 $K$ 에 도달했다면 **가장 낮은 $\Delta$ 를 가진 기존 멤버 축출**

In [ ]:
class CoulombArchive:
    def __init__(self, capacity, gamma, tau=-np.inf):
        self.cap   = capacity
        self.gamma = gamma
        self.tau   = tau
        self.idx   = []  # indices into the global name/hs/q arrays
        self.history = []  # list of (event, target_idx, archive_snapshot)

    def _delta(self, i, archive):
        return q[i] - self.gamma * potential(i, archive)

    def offer(self, i):
        # Compute delta of the candidate relative to current archive
        d_cand = self._delta(i, self.idx)
        if d_cand <= self.tau:
            self.history.append(('reject_threshold', i, list(self.idx)))
            return False
        if len(self.idx) < self.cap:
            self.idx.append(i)
            self.history.append(('admit', i, list(self.idx)))
            return True
        # Full capacity: compare against weakest member (evaluated WITHOUT the candidate)
        weakest_j = min(self.idx, key=lambda j: self._delta(j, [x for x in self.idx if x != j]))
        d_weak = self._delta(weakest_j, [x for x in self.idx if x != weakest_j])
        # Evaluate candidate assuming we would REPLACE the weakest (potential recomputed)
        d_cand_after = self._delta(i, [x for x in self.idx if x != weakest_j])
        if d_cand_after > d_weak:
            self.idx.remove(weakest_j)
            self.idx.append(i)
            self.history.append(('evict_and_admit', (i, weakest_j), list(self.idx)))
            return True
        self.history.append(('reject_full', i, list(self.idx)))
        return False

class ScoreOnlyArchive:
    def __init__(self, capacity):
        self.cap = capacity
        self.idx = []
    def offer(self, i):
        if len(self.idx) < self.cap:
            self.idx.append(i); return True
        weakest = min(self.idx, key=lambda j: q[j])
        if q[i] > q[weakest]:
            self.idx.remove(weakest); self.idx.append(i); return True
        return False

# Ingestion order — intentionally stresses the system:
#   near-duplicate pairs arrive back-to-back to see if the second gets rejected.
ingest_order = ['H4_clean', 'H5_cluttered',     # clutter pair
                'H6_and_ref', 'H7_and_swap',    # AND-swap pair
                'H8_pos', 'H2_causal_ref',
                'H3_causal_rev',                # causal reversal — should coexist with H2
                'H0_stock_baseline', 'H10_correlate',
                'H11_contradict', 'H9_neg', 'H1_weather_baseline']
ingest_idx = [names.index(n) for n in ingest_order]

gamma_default = 0.3   # γ ≥ 0.2 is where near-duplicate rejection kicks in for this dataset (see sweep below)
cap = 5   # room for the diverse semantic categories (finance, indicator, momentum, temperature-corr, temperature-contra, causal)

coulomb = CoulombArchive(capacity=cap, gamma=gamma_default)
score_only = ScoreOnlyArchive(capacity=cap)
for i in ingest_idx:
    coulomb.offer(i)
    score_only.offer(i)

def summarize(archive_idx, label):
    print(f'\n{label}  (n = {len(archive_idx)}):')
    for i in archive_idx:
        print(f'  - {names[i]:<24}  score={q[i]:.2f}')

summarize(coulomb.idx,    f'Coulomb Archive (γ = {gamma_default})')
summarize(score_only.idx, 'Score-only baseline (top-K by score)')

## 6. 붕괴 시나리오 검증 — Coulomb vs Score-only

다음 세 가지가 자동 검증되어야 한다.

- **Clutter**: Coulomb 은 H4/H5 중 하나만, Score-only 는 둘 다
- **AND swap**: Coulomb 은 H6/H7 중 하나만, Score-only 는 둘 다
- **Causal reversal**: Coulomb 은 H2/H3 둘 다 유지 가능 (그들은 서로 안 붐빔)

In [ ]:
def has(archive_idx, name):
    return names.index(name) in archive_idx

def check(label, condition):
    tag = 'PASS' if condition else 'FAIL'
    print(f'  [{tag}] {label}')

print('=== Coulomb Archive ===')
check('Clutter pair — only ONE of H4/H5 kept',
      (has(coulomb.idx, 'H4_clean') + has(coulomb.idx, 'H5_cluttered')) == 1)
check('AND-swap pair — only ONE of H6/H7 kept',
      (has(coulomb.idx, 'H6_and_ref') + has(coulomb.idx, 'H7_and_swap')) == 1)
# Correct semantic check: did Coulomb treat H2/H3 as near-duplicates?
# The right question is whether they were EVER simultaneously admitted, not whether
# they both survive competition against unrelated maximally-isolated candidates.
h2, h3 = names.index('H2_causal_ref'), names.index('H3_causal_rev')
coexisted = any(h2 in snap and h3 in snap for _, _, snap in coulomb.history)
check('Causal reversal — H2 and H3 coexisted at some point (not repelled as duplicates)',
      coexisted)

print('\n=== Score-only baseline ===')
check('Clutter pair — BOTH kept (baseline failure mode)',
      has(score_only.idx, 'H4_clean') and has(score_only.idx, 'H5_cluttered'))
check('AND-swap pair — BOTH kept (baseline failure mode)',
      has(score_only.idx, 'H6_and_ref') and has(score_only.idx, 'H7_and_swap'))

def diversity(archive_idx):
    if len(archive_idx) < 2: return 0.0
    ds = [D[i, j] for i in archive_idx for j in archive_idx if i < j]
    return float(np.mean(ds))

print('\nMean pairwise distance in final archive:')
print(f'  Coulomb   : {diversity(coulomb.idx):.4f}')
print(f'  Score-only: {diversity(score_only.idx):.4f}')
print('\nMean quality:')
print(f'  Coulomb   : {np.mean(q[coulomb.idx]):.4f}')


## 7. Streaming 이벤트 로그

각 자식이 들어올 때 Coulomb 아카이브가 실제로 어떤 판단을 내렸는지 확인.

In [ ]:
print(f'Coulomb streaming log (γ = {gamma_default}, cap = {cap}):')
for k, (event, target, snapshot) in enumerate(coulomb.history):
    if event == 'evict_and_admit':
        admit_i, evict_j = target
        line = f'admit {names[admit_i]:<22} + evict {names[evict_j]:<22}'
    else:
        line = f'{event:<22} {names[target]:<22}'
    snap_str = '[' + ', '.join(names[x].split("_")[0] for x in snapshot) + ']'
    print(f'  step {k+1:2d}: {line}  archive={snap_str}')

## 8. $\gamma$ 감도 스윕

단 하나의 하이퍼파라미터 $\gamma$ 가 exploration-exploitation 사이에서 어떻게 아카이브 구성을 바꾸는지 관찰.

In [ ]:
gammas = np.linspace(0.0, 1.0, 21)
div_list, qual_list, dup_pairs_list = [], [], []

for g in gammas:
    arc = CoulombArchive(capacity=cap, gamma=g)
    for i in ingest_idx:
        arc.offer(i)
    div_list.append(diversity(arc.idx))
    qual_list.append(float(np.mean(q[arc.idx])))
    dup = 0
    for i in arc.idx:
        for j in arc.idx:
            if i < j and K[i, j] > 0.9:
                dup += 1
    dup_pairs_list.append(dup)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(gammas, div_list, 'o-', color='tab:blue', label='mean pairwise distance')
ax1.plot(gammas, qual_list, 's-', color='tab:orange', label='mean quality')
ax1.set_xlabel('γ (repulsion strength)')
ax1.set_ylabel('metric value')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left')
ax2 = ax1.twinx()
ax2.plot(gammas, dup_pairs_list, '^-', color='tab:red', label='near-dup pairs (K > 0.9)')
ax2.set_ylabel('near-duplicate pair count', color='tab:red')
ax2.legend(loc='upper right')
plt.title('γ sweep — single dial controls exploration/exploitation trade-off')
plt.tight_layout(); plt.show()

print(f'{"γ":<8}{"mean dist":>12}{"mean qual":>12}{"near-dup":>10}')
for g, d, ql, dp in zip(gammas[::4], div_list[::4], qual_list[::4], dup_pairs_list[::4]):
    print(f'{g:<8.2f}{d:>12.4f}{ql:>12.4f}{dp:>10d}')

## 9. Potential Field 시각화

가설 공간을 MDS 2D 에 투영하고 그 위에 Coulomb potential 을 등고선으로 그려 **반발 field** 를 눈으로 확인. 등고선이 조밀한 곳은 반발이 강한 exclusion zone, 넓은 골짜기는 새 자식이 편입되기 유리한 지대.

*Note: MDS 공간의 Euclidean 거리는 tree kernel 거리의 근사이므로 이 그림은 참고용. 실제 Coulomb 계산은 tree kernel 거리로 수행됨.*

In [ ]:
def classical_mds(sim, dim=2):
    S = np.clip(sim, 0.0, 1.0)
    D2 = np.maximum(0.0, 2.0 - 2.0 * S)
    n = D2.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J
    w, v = np.linalg.eigh(B)
    order = np.argsort(w)[::-1][:dim]
    return v[:, order] * np.sqrt(np.maximum(w[order], 0.0))

xy = classical_mds(K)

def field_potential(xy_grid, xy_archive, scores_archive, eps=1e-2):
    # Vectorized: for each grid point, sum q_i / (|x - x_i|^2 + eps)
    diff = xy_grid[:, :, None, :] - xy_archive[None, None, :, :]  # (H, W, k, 2)
    dist2 = np.sum(diff ** 2, axis=-1)                            # (H, W, k)
    return np.sum(scores_archive[None, None, :] / (dist2 + eps), axis=-1)  # (H, W)

margin = 0.2
x_min, x_max = xy[:, 0].min() - margin, xy[:, 0].max() + margin
y_min, y_max = xy[:, 1].min() - margin, xy[:, 1].max() + margin
gx, gy = np.meshgrid(np.linspace(x_min, x_max, 220),
                     np.linspace(y_min, y_max, 220))
grid = np.stack([gx, gy], axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
for ax, arc_idx, title in [(axes[0], coulomb.idx,    f'Coulomb Archive (γ={gamma_default})'),
                           (axes[1], score_only.idx, 'Score-only baseline')]:
    arc_xy = xy[arc_idx]
    arc_q  = q[arc_idx]
    U_field = field_potential(grid, arc_xy, arc_q)
    U_log = np.log1p(U_field)
    cs = ax.contourf(gx, gy, U_log, levels=25, cmap='magma')
    ax.contour(gx, gy, U_log, levels=10, colors='white', linewidths=0.35, alpha=0.6)
    # Overlay all hypotheses (gray) then archive members (large)
    ax.scatter(xy[:, 0], xy[:, 1], s=40, c='lightgray', edgecolors='black', zorder=2)
    ax.scatter(arc_xy[:, 0], arc_xy[:, 1], s=280, c='cyan', edgecolors='black',
               marker='X', label='archive members', zorder=3)
    for i, n in enumerate(names):
        ax.annotate(n, (xy[i, 0], xy[i, 1]), fontsize=7,
                    xytext=(5, 5), textcoords='offset points', color='white')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('MDS-1'); ax.set_ylabel('MDS-2')
    ax.legend(loc='lower right')
    plt.colorbar(cs, ax=ax, fraction=0.046, label='log(1 + U)')
plt.tight_layout(); plt.show()

## 10. 요약 관찰

1. **Streaming 아카이빙 성립**: `CoulombArchive.offer(...)` 는 자식이 1개씩 들어올 때마다 즉시 편입/축출 결정. 배치 재선택 필요 없음.
2. **Near-duplicate 자동 축출**: H4 편입 후 H5 는 규칙 2 에 의해 강하게 반발되어 축출. H6/H7 도 마찬가지.
3. **논리 대비는 보존**: H2 와 H3 는 트리 커널 거리에서 이미 멀리 떨어져 있어 서로 반발하지 않음 → 둘 다 아카이브에 살아남을 수 있음.
4. **부모 샘플링 편향 확인**: $\gamma$ 가 증가함에 따라 붐빈 지역(H4, H5, H6, H7)의 선택 확률이 감소하고 격리된 지역(H10, H11)이 상대적 이득.
5. **$\gamma$ 하나의 조작 파워**: 스윕 결과에서 아카이브 다양성 vs 평균 quality 의 트레이드오프가 부드럽게 이동. 실전 튜닝 시 이 곡선 위에서 원하는 지점을 고르면 됨.
6. **Potential field 시각화**: Coulomb 아카이브의 exclusion zone 이 실제로 형성되어 있고, 새 자식이 도착할 만한 '빈 골짜기'를 시각적으로 확인 가능.

### 한계 및 향후 확장

- Atomic 유사도가 n-gram Jaccard 라 실제 SBERT 대비 의미 파악이 약함. 진짜 실험에서는 SBERT 로 교체 권장.
- Tree kernel 이 IMPLIES(atomic, atomic) 형태 가설들에 대해 '구조적 floor' (~0.33 유사도)를 부여함. 이는 진짜 다양성 축이 아닌 형식적 유사성이므로 관찰 시 유념.
- 실전 HypoEvolve 통합 시 `hypoevolve/memory/archive.py` 의 `MAPElitesArchive` 를 상속받아 `CoulombArchive` 로 대체하는 것이 자연스러운 다음 스텝.